In [ ]:
import pandas as pd
import ast
df = pd.read_csv('../../data/transcript_portfolio_profile.csv')

offer completed는 “조건을 만족하는 구매가 이미 발생한 상태”를 의미한다
따라서 completed는 별도의 행동이 아니라 transaction 이후 발생하는 이벤트
조건 충족 구매는 transaction_time <= completed_time으로 봐야 한다

completed 이후 transaction은 추가 구매(업셀/재방문)를 의미한다
따라서 퍼널과 전환 정의는 아래처럼 분리해야 한다:

[프로모션 효과]
received → viewed → completed

[구매 발생]
received → transaction

[추가 소비]
completed → transaction

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 306137 entries, 0 to 306136
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   customer_id       306137 non-null  str    
 1   event             306137 non-null  str    
 2   time              306137 non-null  int64  
 3   offer_id          167184 non-null  str    
 4   amount            138953 non-null  float64
 5   event_reward      33182 non-null   float64
 6   offer_type        167184 non-null  str    
 7   offer_reward      167184 non-null  float64
 8   difficulty        167184 non-null  float64
 9   duration          167184 non-null  float64
 10  channels          167184 non-null  str    
 11  web               167184 non-null  float64
 12  email             167184 non-null  float64
 13  mobile            167184 non-null  float64
 14  social            167184 non-null  float64
 15  gender            272370 non-null  str    
 16  age               272370 non-nu

In [20]:
df.head()

,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on,value
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaT,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26,{'offer id': '2906b810c7d4411798c6938adc9daaa5'}
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaT,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaT,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}


In [19]:
df['event'].value_counts() #[df['offer_type'] == 'informational']

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33182
Name: count, dtype: int64

In [11]:
# 날짜형 변환이 필요하면 사용
df['became_member_on'] = pd.to_datetime(df['became_member_on'], errors='coerce')

# ----------------------------
# 1. 이벤트별 테이블 분리
# ----------------------------
# offer completed 이벤트
completed_df = df[df['event'] == 'offer completed'][
    ['customer_id', 'offer_id', 'time', 'event_reward', 'offer_type', 'difficulty', 'duration']
].copy()

completed_df = completed_df.rename(columns={'time': 'completed_time'})

# transaction 이벤트
transaction_df = df[df['event'] == 'transaction'][
    ['customer_id', 'time', 'amount']
].copy()

transaction_df = transaction_df.rename(columns={'time': 'transaction_time'})

# offer received 이벤트 (참고용)
received_df = df[df['event'] == 'offer received'][
    ['customer_id', 'offer_id', 'time']
].copy()

received_df = received_df.rename(columns={'time': 'received_time'})

# ----------------------------
# 2. completed 기준으로 transaction 연결
# ----------------------------
# 같은 고객 기준으로 transaction 붙이기
check_df = completed_df.merge(
    transaction_df,
    on='customer_id',
    how='left'
)

# ----------------------------
# 3. completed 전/동시/후 transaction 여부 확인
# ----------------------------
# 조건 충족 구매로 볼 수 있는 transaction
check_df['tx_before_or_at_completed'] = (
    check_df['transaction_time'] <= check_df['completed_time']
)

# completed 이후 추가 구매
check_df['tx_after_completed'] = (
    check_df['transaction_time'] > check_df['completed_time']
)

# completed와 transaction 시간 차이
check_df['time_diff'] = check_df['transaction_time'] - check_df['completed_time']

# ----------------------------
# 4. completed 단위로 요약
# ----------------------------
summary = check_df.groupby(
    ['customer_id', 'offer_id', 'completed_time'],
    as_index=False
).agg(
    has_tx_before_or_at=('tx_before_or_at_completed', 'max'),
    has_tx_after=('tx_after_completed', 'max'),
    tx_count=('transaction_time', 'count'),
    min_time_diff=('time_diff', 'min'),
    max_time_diff=('time_diff', 'max')
)

# ----------------------------
# 5. 핵심 확인
# ----------------------------
# print("=== completed 요약 샘플 ===")
# print(summary.head())

print("\n=== completed 기준 transaction 존재 패턴 ===")
print(summary[['has_tx_before_or_at', 'has_tx_after']].value_counts())

print("\n=== completed 이전 또는 같은 시점 transaction 존재 비율 ===")
print(summary['has_tx_before_or_at'].value_counts(normalize=True))

# ----------------------------
# 6. completed와 가장 가까운 transaction 보기
# ----------------------------
closest_tx = check_df.copy()
closest_tx['abs_time_diff'] = closest_tx['time_diff'].abs()

closest_tx = closest_tx.sort_values('abs_time_diff').groupby(
    ['customer_id', 'offer_id', 'completed_time'],
    as_index=False
).first()

print("\n=== completed와 가장 가까운 transaction 샘플 ===")

closest_tx[
    ['customer_id', 'offer_id', 'completed_time', 'transaction_time', 'amount', 'time_diff']
].head(10)


=== completed 기준 transaction 존재 패턴 ===
has_tx_before_or_at  has_tx_after
True                 True            29409
                     False            3773
Name: count, dtype: int64

=== completed 이전 또는 같은 시점 transaction 존재 비율 ===
has_tx_before_or_at
True    1.0
Name: proportion, dtype: float64

=== completed와 가장 가까운 transaction 샘플 ===


,customer_id,offer_id,completed_time,transaction_time,amount,time_diff
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576,576,10.27,0
1,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,414,414,8.57,0
2,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,528,528,14.11,0
3,0011e0d4e6b944f998e987f904e8c1e5,0b1e1539f2cc45b7b9fa7c272da2e1d7,576,576,22.05,0
4,0011e0d4e6b944f998e987f904e8c1e5,2298d6c36e964ae4a3e7e9706d1fb8c2,252,252,11.93,0
5,0011e0d4e6b944f998e987f904e8c1e5,9b98b8c7a33c4b65b9aebfe6a799e6d9,576,576,22.05,0
6,0020c2b971eb4e9188eac86d93036a77,4d5c57ea9a6940dd891ad53e9dbe8da0,510,510,17.24,0
7,0020c2b971eb4e9188eac86d93036a77,fafdcd668e3743c1bb461111dcafc2a4,54,54,17.63,0
8,0020c2b971eb4e9188eac86d93036a77,fafdcd668e3743c1bb461111dcafc2a4,510,510,17.24,0
9,0020ccbbb6d84e358d3414a3ff76cffd,2298d6c36e964ae4a3e7e9706d1fb8c2,222,222,11.65,0
